## Extração pela Gradient Sports API de dados de competições, times, temporadas e jogos para para dataframes

Importa o cliente de `gradient_client.py`.  
A autenticação é lida automaticamente do arquivo `.env` (`BEARER_TOKEN`).

In [1]:
import pandas as pd
import sys
from pathlib import Path

sys.path.append(str(Path().resolve().parent.parent))
from src.gradient_client import GradientSportsClient

pd.set_option('display.max_columns', None)

In [2]:
def camel_case_dotted_columns(df: pd.DataFrame) -> pd.DataFrame:
    """
    Renomeia colunas com ponto (ex: "team.id", "competition.name") para
    camelCase sem ponto (ex: "teamId", "competitionName") — nomes com ponto
    são inválidos/ambíguos em ferramentas downstream (ex: Spark trata "." como
    separador de struct).
    """
    def to_camel(col):
        if "." not in col:
            return col
        parts = col.split(".")
        return parts[0] + "".join(p[:1].upper() + p[1:] for p in parts[1:])

    return df.rename(columns={c: to_camel(c) for c in df.columns})

### Status da requisição

In [3]:
client = GradientSportsClient()

# health check
client.get_status()

{'data': {'status': 'ok'}}

### Competições

In [4]:
# competitions & seasons the account can access
df_competitions = client.get_competitions(as_dataframe=True).drop('datasets', axis=1).drop_duplicates().reset_index(drop=True)
df_competitions = camel_case_dotted_columns(df_competitions)

df_competitions

,season,competitionId,competitionName
0,2020-2021,1,Premier League
1,2021-2022,1,Premier League
2,2022-2023,1,Premier League
3,2023-2024,1,Premier League
4,2024-2025,1,Premier League
5,2025-2026,1,Premier League
6,2026-2027,1,Premier League
7,2023,42,Brasileiro Série A
8,2024,42,Brasileiro Série A
9,2025,42,Brasileiro Série A


In [5]:
df_competitions.to_csv(str(Path().resolve().parent.parent / "data" / "competitions.csv"), index=False)

### Times

In [6]:
# teams the account can access
df_teams = client.get_teams(as_dataframe=True).drop('dataset', axis=1).drop_duplicates()
df_teams = camel_case_dotted_columns(df_teams).sort_values('competitionId').reset_index(drop=True)

df_teams

,teamId,teamName,competitionId,competitionName
0,1.0,AFC Bournemouth,1,Premier League
1,20.0,Wolverhampton Wanderers,1,Premier League
2,221.0,Nottingham Forest,1,Premier League
3,335.0,Sunderland AFC,1,Premier League
4,10.0,Liverpool,1,Premier League
...,...,...,...,...
56,437.0,Fortaleza,42,Brasileiro Série A
57,514.0,Cruzeiro,42,Brasileiro Série A
58,864.0,Chapecoense,42,Brasileiro Série A
59,517.0,Bahia,42,Brasileiro Série A


In [7]:
df_teams.to_csv(str(Path().resolve().parent.parent / "data" / "teams.csv"), index=False)

### Jogos

In [19]:
# all games — or filter by season / competition / team
# as_dataframe=True → one row per game, nested dicts dot-expanded
df_games = client.get_games(as_dataframe=True)

df_games = camel_case_dotted_columns(df_games)

df_games = df_games.rename(columns={"id":"gameId"})

df_games.drop('updatedAt', axis=1, inplace=True)

df_games

,gameId,date,season,teamExtraTimeStartSide,teamStartSide,venueType,teamId,teamName,competitionId,competitionName,opponentTeamId,opponentTeamName,stadiumName,stadiumLength,stadiumWidth
0,51624,2026-04-01,2026,Right,Right,TEAM_HOME,430,Botafogo,42,Brasileiro Série A,868,Mirassol FC,Estádio Olímpico Nilton Santos,105.0,68.0
1,51649,2026-04-11,2026,Left,Left,OPPONENT_HOME,517,Bahia,42,Brasileiro Série A,868,Mirassol FC,Estádio José Maria de Campos Maia,105.0,68.0
2,13383,2023-09-16,2023-2024,Left,Right,OPPONENT_HOME,11,Manchester City,1,Premier League,19,West Ham,London Stadium,105.0,68.0
3,24338,2024-11-23,2024,Right,Right,TEAM_HOME,436,Fluminense FC,42,Brasileiro Série A,437,Fortaleza,Maracanã,105.0,68.0
4,51698,2026-05-17,2026,Right,Right,TEAM_HOME,864,Chapecoense,42,Brasileiro Série A,1075,Remo,Arena Condá,105.0,68.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3648,490,2021-04-19,2020-2021,Right,Left,TEAM_HOME,55,Leeds United,1,Premier League,10,Liverpool,Elland Road,105.0,68.0
3649,41057,2025-12-30,2025-2026,Left,Right,OPPONENT_HOME,4,Brighton & Hove Albion,1,Premier League,19,West Ham,London Stadium,105.0,68.0
3650,4548,2022-10-19,2022-2023,Right,Left,TEAM_HOME,119,Brentford,1,Premier League,6,Chelsea,Gtech Community Stadium,105.0,68.0
3651,41004,2025-12-03,2025-2026,Right,Right,TEAM_HOME,10,Liverpool,1,Premier League,335,Sunderland AFC,Anfield,101.0,68.0


In [20]:
df_games.to_csv(str(Path().resolve().parent.parent / "data" / "games.csv"), index=False)